In [1]:
from __future__ import annotations

import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor as PoolExecutor, as_completed
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

# ===========================
# Config (EDIT THESE PATHS)
# ===========================
WORK_ROOT    = Path(r"C:\Temp_Instru")
INPUT_CLONES = WORK_ROOT / "clonesV9.0"
OUTPUT_MINE  = WORK_ROOT / "mineV9.3"

# Study end date (UTC)
CUTOFF_ISO = "2025-08-10 23:59:59 +0000"

# Timeline scope for evolution analysis:
#   "repo_wide" -> walk all origin/* heads (recommended for RQ3)
#   "default"   -> only default branch history
TIMELINE_SCOPE = "repo_wide"

# Performance / behavior toggles
MAX_REPOS = 0  # 0 = all
MAX_WORKERS = min(32, (os.cpu_count() or 8) * 2)
RESUME_IF_EXISTS = True
SUPPRESS_EMPTY_ROWS = True
CAPTURE_EMPTY_GAPS = True
EMIT_NONE_STATE = False

# Blob size guards
BLOB_MAX_SIZE = 2_000_000
HARD_MAX_SIZE = 5_000_000

# High-priority CI paths never skipped (even if large)
HIGH_PRIORITY_CI_PATHS = {
    ".github/workflows",
    ".gitlab-ci.yml",
    "azure-pipelines.yml",
    ".circleci/config.yml",
    ".bitrise.yml",
    ".travis.yml",
}

# --- Add this near the top of your miner (config area) ---
# Folder names of the few problematic repos (as they appear under INPUT_CLONES)
PROBLEM_REPO_NAMES = {
    "MetaMask__metamask-mobile",
    "rainbow-me__rainbow",
    "pytorch__executorch"   # <-- replace with the actual name(s)
}


# ===========================
# Quick startup checks
# ===========================
def _failfast_checks() -> None:
    if shutil.which("git") is None:
        print("[fatal] Git not found in PATH. Install Git and/or add it to PATH.", file=sys.stderr)
        raise SystemExit(1)
    if not WORK_ROOT.exists():
        print(f"[fatal] WORK_ROOT does not exist: {WORK_ROOT}", file=sys.stderr)
        raise SystemExit(1)
    if not INPUT_CLONES.exists():
        print(f"[warn] INPUT_CLONES does not exist yet: {INPUT_CLONES}")
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

# ===========================
# File classifiers
# ===========================
YAML_EXTS = (".yml", ".yaml")
GRADLE_NAMES = {"build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts"}
GRADLE_EXTS = (".gradle", ".gradle.kts")
SCRIPT_EXTS = (".sh", ".bat", ".cmd", ".ps1")
CI_BUILD_SPECIAL = {"Jenkinsfile", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml"}
XML_BUILD_FILES = {"pom.xml", "build.xml", "config.xml"}

def is_yaml(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in YAML_EXTS or os.path.basename(path) in CI_BUILD_SPECIAL

def is_gradle(path: str) -> bool:
    name = os.path.basename(path)
    ext = os.path.splitext(path)[1].lower()
    return name in GRADLE_NAMES or ext in GRADLE_EXTS

def is_script(path: str) -> bool:
    ext = os.path.splitext(path)[1].lower()
    return ext in SCRIPT_EXTS or os.path.basename(path) in ("Jenkinsfile",)

def is_ci_xml_or_build_xml(path: str) -> bool:
    return os.path.basename(path).lower() in {n.lower() for n in XML_BUILD_FILES}

def is_relevant_file(path: str) -> bool:
    return is_yaml(path) or is_gradle(path) or is_script(path) or is_ci_xml_or_build_xml(path)

def is_high_priority_ci_path(path: str) -> bool:
    norm = path.replace("\\", "/")
    base = os.path.basename(norm)
    if base in HIGH_PRIORITY_CI_PATHS:
        return True
    for root in HIGH_PRIORITY_CI_PATHS:
        if norm.startswith(root.rstrip("/") + "/"):
            return True
    return False

# ===========================
# Regex helpers & normalizers
# ===========================
COMMENT_LINE_RE = re.compile(r"(?m)^\s*(#|//|REM\b|::).*?$")

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# --- Gradle comment stripper (preserves http(s)://) ---
def strip_comments_gradle(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.S)                 # /* ... */
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.M)                  # // ...
    return s

def normalize_block_keys(text: str) -> str:
    # expose content of run/script/command keys and drop YAML dashes before likely shell lines
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$", r"\2", text)
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$", "", text)
    text = re.sub(
        r"(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)",
        r"\1",
        text,
    )
    return text

IGNORE_GHA_ACTIONS_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*(docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)|actions/checkout|docker/setup-qemu-action|docker/setup-buildx-action)@.*$"
)

def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub("", text or "")

GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def pre_sanitize(text: str) -> str:
    return GHA_EXPR_RE.sub("", text or "")

# Gradle command shapes (for CI scripts/YAML)
GRADLE_PREFIX = (
    r"^\s*"
    r"(?:\S+=\S+\s+)*"
    r"(?:sudo\s+)?"
    r"(?:(?:bash|sh)\s+-c[l]?\s+[\'\"]?)?"
    r"(?:[^#\n;]*?&&\s+)?"
    r"(?:cd\s+\S+\s+&&\s+)?"
    r"(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?"
)
GRADLE_ANYWHERE = r"(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*"
GRADLE_ANYWHERE_RE = re.compile(GRADLE_ANYWHERE)
GRADLE_BUILD_ACTION_RE = re.compile(r"(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@")
NON_TEST_PREFIX = r"(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)"
SHELL_PREFIX = r"(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'\"]?)?(?:[^#\n;]*?&&\s+)?"

# Device / env patterns (CI side)
EMULATOR_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+"
ADB_WAIT_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b"
ADB_SERIAL_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)"
REAL_DEVICE_LINE = rf"(?mi)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b"

def compile_any(patterns: List[str], flags: int = re.I | re.M) -> List[re.Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable, text: str) -> bool:
    for p in patterns:
        if isinstance(p, str):
            p = re.compile(p, re.I | re.M)
        if p.search(text):
            return True
    return False

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# -------- DEVICE SOURCES --------
DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [REAL_DEVICE_LINE]),
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device", [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@", [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r"(?mi)^\s*(?:\./)?android-wait-for-emulator\b"]),
    ("Emulator", "start-emulator.sh", [r"(?mi)^\s*start-emulator\.sh\b"]),
    ("Emulator", "android create avd", [r"\bandroid\b[^\n]*\bcreate\s+avd\b"]),
    ("Emulator", "circleci android orb", [
        r"(?mi)^\s*(?:-\s*)?android/start-emulator-and-run-tests\s*:",
        r"(?mi)^\s*system-image\s*:\s*system-images;android-\d+;google_apis;"
    ]),
    ("Emulator", "reactivecircus runner", [r"(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+"]),
    ("Emulator", "malinskiy runner", [r"(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+"]),
    ("Emulator", "sys-img component", [
        r"(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b",
        r"(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "avdmanager", [r"(?m)^\s*\S*avdmanager\b"]),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r"^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "other gha emulator", [
        r"(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+",
        r"(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)(?!malinskiy/action-android/emulator-run-cmd@)(?!emulator-wtf/run-tests@)[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+"
    ]),
    ("Third_Party_Lab", "gcloud firebase", [r"(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "browserstack/bstack", [r"(?i)\b(browserstack|bstack)\b"]),
    ("Third_Party_Lab", "appcenter test", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "maestro cloud", [r"(?mi)^[^\n]*\bmaestro\s+cloud\b"]),
    ("Third_Party_Lab", "emulator.wtf action", [
        r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+",
        r"(?i)\bemulator\.wtf\b"
    ]),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

# -------- Triggers (CI/YAML side) --------
TRIGGER_SOURCES_PRIMARY = [
    ("Gradle", "connectedAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*"]),
    ("Gradle", "connected.*Android.*", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b(?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)[^\n\r]*"]),
    ("Gradle", "connectedCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*"]),
    ("Gradle", "cAT shorthand", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*"]),
    ("Gradle", "deviceCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*"]),
    ("Gradle", "managedDevice AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "variant/device AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "Spoon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b"]),
    ("ADB", "am instrument", [r"(?mi)^[^\n]*\bam\s+instrument\b"]),
    ("Third_Party_Lab", "gcloud firebase (instr)", [r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "appcenter", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "emulator.wtf run", [r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+", r"(?i)\bemulator\.wtf\b"]),
]
TRIGGER_SOURCES_PRIMARY += [
    ("Gradle", "generateBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "collectBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "connectedBenchmarkAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*"]),
]

TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b"]),
    ("Gradle", "connectedAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b"]),
    ("Gradle", "connectedCheck (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connectedcheck\b"]),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b"]),
    ("Gradle", "variant/device AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})[\w-]*androidtest\b"]),
    ("Gradle", "Spoon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b"]),
]
TRIGGER_SOURCES_ANYWHERE += [
    ("Gradle", "generateBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bgenerate(?:\w*?)baselineprofile\b"]),
    ("Gradle", "collectBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bcollect(?:\w*?)baselineprofile\b"]),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedbenchmarkandroidtest\b"]),
]

TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

def collect_hits_with_groups(patterns, text: str):
    labels = []
    groups = []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# GHA gradle inputs guard
GHA_GRADLE_INPUTS = compile_any([
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:(?:[:\w-]+:)*) (?!assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)[\w-]*androidtest\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b",
])

PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@", r"(?i)\bemulator\.wtf\b"])),
    ("firebase-test-lab", compile_any([r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b", r"(?mi)\bflank\s+android\s+run\b"])),
    ("browserstack", compile_any([r"(?i)\bbrowserstack\b", r"(?i)\bbstack\b"])),
    ("sauce-labs", compile_any([r"(?mi)\bsaucectl(?:\s+run)?\b", r"(?mi)\bsauce\s+ctl\b"])),
    ("appcenter", compile_any([r"(?mi)\bappcenter\s+test\s+run\s+android\b"])),
    ("maestro-cloud", compile_any([r"(?mi)\bmaestro\s+cloud\b"])),
]

INLINE_DEVICE_HINTS = compile_any([
    r"(?mi)^\s*devices\s*:\s*\|",
    r"(?mi)\b--device\b",
    r"(?mi)\bmodel\s*=\s*[^,\s]+",
    r"(?mi)\bversion\s*=\s*\d+",
    r"(?mi)\blocale\s*=\s*[-\w]+",
    r"(?mi)\borientation\s*=\s*(portrait|landscape)",
    r"(?mi)^\s*with-orchestrator\s*:\s*true\b",
    r"(?mi)\b--use-orchestrator\b",
    r"(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b",
    r"(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b",
])

CONFIG_FILE_HINTS = compile_any([
    r"(?mi)\.ewtf\.ya?ml\b",
    r"(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b",
    r"(?mi)\b--config(?:=|\s+)\S+",
    r"(?mi)\b(browserstack\.ya?ml)\b",
    r"(?mi)\b(bs(?:config)?\.ya?ml)\b",
])

ANDROID_CONTEXT_RE = re.compile(
    r"(?i)\b(adb|avd|emulator|android\s+sdk|system-images;android-|androidtest|connected(?:check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run)\b"
)

EMULATOR_COMMUNITY_LABELS = {"reactivecircus runner", "malinskiy runner", "other gha emulator", "circleci android orb"}
EMULATOR_CUSTOM_LABELS = {
    "adb -s emulator-serial",
    "adb wait-for-device",
    "emulator -avd/@",
    "android-wait-for-emulator",
    "start-emulator.sh",
    "android create avd",
    "avdmanager",
    "sdkmanager system-images/emulator",
    "sys-img component",
}
THIRD_PARTY_STRONG_LABELS = {"gcloud firebase", "emulator.wtf run", "saucectl", "appcenter", "browserstack/bstack", "maestro cloud"}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
STRONG_DEVICE_LABELS = EMULATOR_COMMUNITY_LABELS | EMULATOR_CUSTOM_LABELS | THIRD_PARTY_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def has_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def has_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

# ===========================
# Gradle GMD config detector (aligned with V6.0)
# ===========================
TESTOPTIONS_HEAD = re.compile(r"\btestOptions\s*\{", re.I)
MANAGED_HEAD     = re.compile(r"\bmanagedDevices\s*\{", re.I)
GMD_INNER_TOKENS_RX = re.compile(r"\b(managedDevices|managedVirtualDevice|devices|groups|deviceGroups)\b", re.I)

def _find_block_span_from_head(text: str, head_start: int) -> Optional[Tuple[int, int]]:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

def find_gmd_block(text_gradle_no_comments: str) -> bool:
    """Return True if we see testOptions{ managedDevices{ … } } or a direct managedDevices{ … } block."""
    t = text_gradle_no_comments
    # Direct managedDevices { ... }
    for m in MANAGED_HEAD.finditer(t):
        if _find_block_span_from_head(t, m.start()):
            return True
    # testOptions { ... } containing GMD tokens
    for m in TESTOPTIONS_HEAD.finditer(t):
        ci = _find_block_span_from_head(t, m.start())
        if not ci:
            continue
        a, b = ci
        sub = t[a:b+1]
        if GMD_INNER_TOKENS_RX.search(sub):
            return True
    return False

# ===========================
# Git helpers
# ===========================
#Prolematic repos set helpers
# --- Keep your current list_repos as-is, but ADD these helpers ---

def list_repos_depth_limited(root: Path, max_depth: int = 2) -> list[Path]:
    """
    Very shallow discovery: scans root, root/*, root/*/* only.
    Avoids deep os.walk on Windows (helps with WinError 206).
    """
    repos: list[Path] = []
    # depth 0
    if (root / ".git").exists():
        return [root]

    # depth 1
    for d1 in root.iterdir():
        if not d1.is_dir():
            continue
        if (d1 / ".git").exists():
            repos.append(d1)
            continue

        if max_depth >= 2:
            # depth 2
            for d2 in d1.iterdir():
                if d2.is_dir() and (d2 / ".git").exists():
                    repos.append(d2)

    repos.sort(key=lambda p: p.name.lower())
    return repos


def list_repos_hybrid(root: Path,
                      problem_repo_names: set[str] | None = None) -> list[Path]:
    """
    Uses your original deep discovery for normal repos,
    and a shallow, direct lookup for the problematic few.
    """
    problem_repo_names = set(problem_repo_names or [])

    # 1) Use your existing discovery first
    deep_found = set()          # names we already found
    try:
        base = list_repos(root)  # <-- this calls your ORIGINAL function
    except Exception:
        base = []

    for p in base:
        deep_found.add(p.name)

    # 2) For each problem repo, add it directly (no os.walk)
    extras: list[Path] = []
    for name in problem_repo_names:
        if name in deep_found:
            continue
        candidate = root / name
        if (candidate / ".git").exists():
            extras.append(candidate)

    # 3) Merge & sort
    merged = base + extras
    merged = sorted({p.resolve() for p in merged}, key=lambda p: p.name.lower())
    return list(merged)






def run_git(repo: Path, args: List[str], text: bool = True, check: bool = True, retries: int = 2) -> subprocess.CompletedProcess:
    last_exc: Optional[BaseException] = None
    for attempt in range(retries + 1):
        try:
            return subprocess.run(
                ["git", "-C", str(repo)] + args,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=text,
                check=check,
                encoding="utf-8" if text else None,
                errors="replace" if text else None,
            )
        except subprocess.CalledProcessError as e:
            last_exc = e
            time.sleep(0.1 * (attempt + 1) + random.random() * 0.1)
        except Exception as e:
            last_exc = e
            time.sleep(0.05)
    if isinstance(last_exc, subprocess.CalledProcessError) and not check:
        return last_exc  # type: ignore[return-value]
    raise last_exc  # type: ignore[misc]

def list_repos(root: Path) -> List[Path]:
    out: List[Path] = []
    for p, dirs, _files in os.walk(root):
        pth = Path(p)
        if (pth / ".git").exists():
            out.append(pth)
            dirs[:] = []
    return out

def default_branch_ref(repo: Path) -> Optional[str]:
    try:
        ref = run_git(repo, ["symbolic-ref", "refs/remotes/origin/HEAD"]).stdout.strip()
        if ref.startswith("refs/remotes/"):
            parts = ref.split("/")
            return "/".join(parts[2:])
    except Exception:
        pass
    for cand in ("origin/main", "origin/master"):
        cp = run_git(repo, ["rev-parse", "--verify", cand], check=False)
        if cp.returncode == 0:
            return cand
    return None

def cutoff_head_default(repo: Path, cutoff_iso: str) -> List[str]:
    ref = default_branch_ref(repo)
    if not ref:
        return []
    cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", ref], check=False)
    sha = (cp.stdout or "").strip()
    return [sha] if sha else []

def cutoff_heads_all(repo: Path, cutoff_iso: str) -> List[str]:
    refs_raw = run_git(repo, ["for-each-ref", "--format=%(refname:short)", "refs/remotes/origin"]).stdout
    refs = [r for r in refs_raw.splitlines() if r and r != "origin/HEAD" and not r.startswith("origin/pr/")]
    shas: List[str] = []
    for r in refs:
        cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", r], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    if not shas:
        cp = run_git(repo, ["rev-list", "-n1", "--before", cutoff_iso, "--all"], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    return shas

def _pathspecs_yaml() -> List[str]:
    return [":(glob)**/*.yml", ":(glob)**/*.yaml", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml", ".bitrise.yml"]

def _pathspecs_gradle() -> List[str]:
    return ["build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts", ":(glob)**/*.gradle", ":(glob)**/*.gradle.kts"]

def _pathspecs_scripts_and_xml() -> List[str]:
    return ["Jenkinsfile", ":(glob)**/*.sh", ":(glob)**/*.bat", ":(glob)**/*.cmd", ":(glob)**/*.ps1", ":(glob)**/pom.xml", ":(glob)**/build.xml", ":(glob)**/config.xml"]

def commits_touching_relevant(repo: Path, heads: List[str]) -> List[str]:
    pathspecs = _pathspecs_yaml() + _pathspecs_gradle() + _pathspecs_scripts_and_xml()
    if not heads:
        return []
    s = run_git(repo, ["rev-list", "--reverse"] + heads + ["--"] + pathspecs).stdout
    return [c for c in s.splitlines() if c.strip()]

def list_tree_entries(repo: Path, treeish: str) -> List[Tuple[str, str]]:
    try:
        raw = run_git(repo, ["ls-tree", "-r", "-z", treeish]).stdout
    except Exception:
        raw = ""
    out: List[Tuple[str, str]] = []
    for entry in raw.split("\x00"):
        if not entry or "\t" not in entry:
            continue
        meta, path = entry.split("\t", 1)
        parts = meta.split()
        if len(parts) < 3:
            continue
        sha = parts[2]
        out.append((sha, path))
    return out

def blob_size(repo: Path, sha: str) -> int:
    try:
        return int(run_git(repo, ["cat-file", "-s", sha]).stdout.strip())
    except Exception:
        return 0

@lru_cache(maxsize=250_000)
def read_blob_cached(repo_path: str, sha: str) -> Optional[str]:
    repo = Path(repo_path)
    try:
        return run_git(repo, ["cat-file", "-p", sha]).stdout
    except Exception:
        return None

# ===========================
# Detection at snapshot level
# ===========================
def scan_snapshot_for_styles(repo: Path, treeish: str) -> Tuple[Set[str], Dict]:
    entries = list_tree_entries(repo, treeish)

    # Aggregate features
    features = {
        "yaml_loc": 0,
        "gradle_loc": 0,
        "runs_on": set(),
        "matrix_width_hint": 0,
        "api_levels": set(),
        "reliability_flags": set(),
        "gmd_config_present": False,
        "gmd_identifiers": set(),  # optional, we only store names when we parse them from block heads
    }
    all_text_blocks: List[Tuple[str, str]] = []
    raw_blocks_for_guard: List[str] = []
    has_gradle_anywhere_repo = False

    RX_RUNS_ON = re.compile(r"(?mi)^\s*runs-on\s*:\s*(.+)$")
    RX_LIST_ITEM = re.compile(r"(?mi)^\s*-\s")
    RX_API_LEVEL = re.compile(r"(?i)\bandroid[-_ ]?(\d{2})\b|system-images;android-(\d{2})\b")
    RX_RETRY = re.compile(r"(?i)\bretry\b|\bmax-attempts\b|\bflaky\b")
    RX_TIMEOUT = re.compile(r"(?i)\btimeout[- ]?minutes\b|\btimeout\b")
    RX_HEADLESS = re.compile(r"(?i)\bheadless\b|\b-no-window\b|\b-no-boot-anim\b")
    RX_WAIT_FOR_DEVICE = re.compile(r"(?i)\badb\s+wait[- ]?for[- ]?device\b")

    # Optional identifier capture in GMD blocks (names before "(ManagedVirtualDevice)")
    GMD_BLOCK_NAME_RX = re.compile(
        r'(\w+)\s*\(\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
        re.I
    )

    for sha, path in entries:
        if not is_relevant_file(path):
            continue
        size = blob_size(repo, sha)
        if not is_high_priority_ci_path(path) and BLOB_MAX_SIZE and size > BLOB_MAX_SIZE:
            continue
        text = read_blob_cached(str(repo), sha)
        if not text or len(text) > HARD_MAX_SIZE:
            continue

        lines = text.splitlines()
        norm = text  # default

        if is_yaml(path):
            features["yaml_loc"] += len(lines)
            for m in RX_RUNS_ON.finditer(text):
                features["runs_on"].add(m.group(1).strip())
            features["matrix_width_hint"] = max(features["matrix_width_hint"], len(RX_LIST_ITEM.findall(text)))
            # sanitize
            norm = strip_comments(norm)
            norm = normalize_block_keys(norm)
            norm = strip_irrelevant_ci_lines(norm)
            norm = pre_sanitize(norm)

        elif is_script(path) or is_ci_xml_or_build_xml(path):
            norm = strip_comments(norm)
            norm = normalize_block_keys(norm)
            norm = strip_irrelevant_ci_lines(norm)
            norm = pre_sanitize(norm)

        elif is_gradle(path):
            features["gradle_loc"] += len(lines)
            # For Gradle, use Gradle-aware comment stripping then look for GMD config blocks
            gradle_clean = strip_comments_gradle(text)
            # Detect managedDevices presence (aligned with your V6.0)
            if not features["gmd_config_present"] and find_gmd_block(gradle_clean):
                features["gmd_config_present"] = True
                # best-effort: capture identifiers (optional, for diagnostics only)
                for m in GMD_BLOCK_NAME_RX.finditer(gradle_clean):
                    ident = (m.group(1) or "").strip()
                    if ident:
                        features["gmd_identifiers"].add(ident)
            # Keep norm as lowercased “clean” text for any incidental signals
            norm = gradle_clean

        # Common feature mining from the (possibly sanitized) text
        for m in RX_API_LEVEL.finditer(text):
            for g in m.groups():
                if g and g.isdigit():
                    features["api_levels"].add(int(g))
        if RX_RETRY.search(text):
            features["reliability_flags"].add("retry")
        if RX_TIMEOUT.search(text):
            features["reliability_flags"].add("timeout")
        if RX_HEADLESS.search(text):
            features["reliability_flags"].add("headless")
        if RX_WAIT_FOR_DEVICE.search(text):
            features["reliability_flags"].add("wait_for_device")

        all_text_blocks.append((path, norm))
        raw_blocks_for_guard.append(norm)

        if GRADLE_ANYWHERE_RE.search(norm) or GRADLE_BUILD_ACTION_RE.search(norm):
            has_gradle_anywhere_repo = True

    # Collect triggers & devices repo-wide (mostly from YAML/scripts)
    trigger_labels: List[str] = []
    trigger_groups: List[str] = []
    device_labels: List[str] = []
    device_groups: List[str] = []

    for _path, norm in all_text_blocks:
        nlow = norm.lower()

        t_labels, t_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, nlow)
        fb_labels, fb_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, nlow)
        if fb_labels:
            t_labels = unique_preserve(t_labels + fb_labels)
            t_groups = unique_preserve(t_groups + fb_groups)

        # GHA gradle input guard
        if any_match(GHA_GRADLE_INPUTS, nlow):
            gha_tied = re.search(
                r"(?mi)^\s*uses\s*:\s*(reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd)@",
                nlow,
            )
            if has_gradle_anywhere_repo or gha_tied:
                if "gha gradle inputs/script" not in t_labels:
                    t_labels.append("gha gradle inputs/script")
                if "Gradle" not in t_groups:
                    t_groups.append("Gradle")

        trigger_labels = unique_preserve(trigger_labels + t_labels)
        trigger_groups = unique_preserve(trigger_groups + t_groups)

        d_labels, d_groups = collect_hits_with_groups(DEVICE_PATTERNS, nlow)
        # Guard: “other gha emulator” must have Android context or a trigger
        if ("other gha emulator" in d_labels) and (not ANDROID_CONTEXT_RE.search(nlow)) and (not t_labels):
            d_labels = [l for l in d_labels if l != "other gha emulator"]
            if not d_labels:
                d_groups = [g for g in d_groups if g != "Emulator"]

        device_labels = unique_preserve(device_labels + d_labels)
        device_groups = unique_preserve(device_groups + d_groups)

    has_test_trigger = bool(trigger_labels)

    # Weak hint suppression: require strong device OR a test trigger
    strong_seen = bool(set(device_labels) & STRONG_DEVICE_LABELS)
    if not (strong_seen or has_test_trigger):
        device_labels = [l for l in device_labels if l in STRONG_DEVICE_LABELS]
        if not device_labels:
            device_groups = []

    # BrowserStack upload-only guard
    repo_text = "\n".join(raw_blocks_for_guard)
    provider = detect_provider(repo_text)
    inline_decl = has_inline_env(repo_text)
    config_decl = has_config_env(repo_text)
    has_ftl_instr = bool(re.search(r"(?mi)\bgcloud\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)", repo_text))
    bs_test_hint = re.search(r"(?mi)\b(browserstack|bstack)\b[^\n]*\b(espresso|instrumentation|app-automate|automate|--device|--devices)\b", repo_text)
    if ("browserstack/bstack" in device_labels or provider == "browserstack"):
        if (not has_test_trigger) and (not inline_decl) and (not config_decl) and (not has_ftl_instr) and (not bs_test_hint):
            device_labels = [l for l in device_labels if l != "browserstack/bstack"]
            if "Third_Party_Lab" in device_groups and not any(l in {"gcloud firebase", "saucectl", "appcenter", "maestro cloud", "emulator.wtf action"} for l in device_labels):
                device_groups = [g for g in device_groups if g != "Third_Party_Lab"]

    # Map to final styles
    styles: Set[str] = set()

    if "Third_Party_Lab" in device_groups:
        styles.add("ThirdParty")

    if "Emulator" in device_groups:
        lbls = set(device_labels)
        if lbls & EMULATOR_COMMUNITY_LABELS:
            styles.add("Emu_Community")
        elif lbls & EMULATOR_CUSTOM_LABELS:
            styles.add("Emu_Custom")

    # --- GMD detection parity with shallow snapshot ---
    # Count GMD if EITHER:
    #  (A) GMD Gradle configuration is present, OR
    #  (B) CI shows managed-device AndroidTest triggers (execution evidence).
    def has_gmd_gradle_trigger(tlabs: List[str]) -> bool:
        L = {l.lower() for l in tlabs}
        if any("manageddevice androidtest" in l for l in L):
            return True
        if ("variant/device androidtest" in L) and not any(x in L for x in {"connected.*android.*", "connectedandroidtest", "connectedbenchmarkandroidtest", "spoon", "marathon"}):
            return True
        return False

    if features["gmd_config_present"] or has_gmd_gradle_trigger(trigger_labels):
        styles.add("GMD")

    meta = {
        "sources": {"yaml": [], "gradle": [], "scripts_xml": []},
        "features": {
            "yaml_loc": features["yaml_loc"],
            "gradle_loc": features["gradle_loc"],
            "runs_on": sorted(features["runs_on"]),
            "matrix_width_hint": features["matrix_width_hint"],
            "api_levels": sorted(int(x) for x in features["api_levels"]),
            "reliability_flags": sorted(features["reliability_flags"]),
            "trigger_labels": sorted(trigger_labels),
            "device_labels": sorted(device_labels),
            "provider": provider,
            "inline_env": bool(inline_decl),
            "config_env": bool(config_decl),
            "gradle_present_repo": bool(has_gradle_anywhere_repo),
            "gmd_config_present": bool(features["gmd_config_present"]),
            "gmd_identifiers": sorted(features["gmd_identifiers"]) if features["gmd_identifiers"] else [],
        },
    }
    return styles, meta

# ===========================
# Timeline building
# ===========================
def empty_result(repo: Path, cutoff_iso: str) -> Dict:
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_default_head": [],
        "cutoff_all_heads": [],
        "first_commit_date": None,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": [],
        "events": {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []},
        "snapshot_as_of_cutoff_default": [],
        "snapshot_as_of_cutoff_repo_wide": [],
        "off_default_extra_styles": [],
        "has_off_default_extra": 0,
        "empty_gaps": [],
        "qa_issue": "No environment detected at cutoff",
    }

def earliest_commit(repo: Path) -> Optional[str]:
    try:
        s = run_git(repo, ["rev-list", "--max-parents=0", "--all"]).stdout.strip()
        return s.splitlines()[0] if s else None
    except Exception:
        return None

def get_commit_date_iso(repo: Path, commit: str) -> Optional[str]:
    try:
        return run_git(repo, ["show", "-s", "--format=%cI", commit]).stdout.strip() or None
    except Exception:
        return None

def union_snapshot_styles_at_cutoff(repo: Path, heads: List[str]) -> List[str]:
    styles: Set[str] = set()
    for sha in heads:
        s, _m = scan_snapshot_for_styles(repo, sha)
        styles |= s
    return sorted(styles)

def build_timeline(repo: Path, cutoff_iso: str) -> Dict:
    # Heads at cutoff
    default_heads = cutoff_head_default(repo, cutoff_iso)
    all_heads     = cutoff_heads_all(repo, cutoff_iso)

    if not default_heads and not all_heads:
        return empty_result(repo, cutoff_iso)

    # Commits touching relevant files on default and repo-wide (for tagging & scope)
    commits_default = commits_touching_relevant(repo, default_heads) if default_heads else []
    commits_all     = commits_touching_relevant(repo, all_heads) if all_heads else commits_default

    # Date maps
    commit_dates: Dict[str, str] = {}
    for c in commits_all:
        d = get_commit_date_iso(repo, c)
        if d:
            commit_dates[c] = d
    first_commit_date = commit_dates.get(commits_all[0]) if commits_all else None

    # Build timeline (scope-controlled)
    scope_commits = commits_all if TIMELINE_SCOPE == "repo_wide" else commits_default
    scope_commits = scope_commits or commits_all  # fallback

    timeline: List[Dict] = []
    events: Dict[str, List[Dict]] = {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []}
    prev: Set[str] = set()

    empty_gaps: List[Dict] = []
    gap_open: Optional[Dict] = None

    # (Optional) historical baseline at repository root
    root = earliest_commit(repo)
    if root:
        base_styles, base_meta = scan_snapshot_for_styles(repo, root)
        if base_styles:
            baseline_date = get_commit_date_iso(repo, root)
            timeline.append({"date": baseline_date, "commit": root, "on_default": int(root in commits_default), "styles": sorted(base_styles), **base_meta})
            for s in sorted(base_styles):
                if s in events:
                    events[s].append({"event": "added", "date": baseline_date, "commit": root, "on_default": int(root in commits_default)})
            prev = set(base_styles)

    default_set = set(commits_default)

    for c in scope_commits:
        styles_now, meta_now = scan_snapshot_for_styles(repo, c)
        dt = commit_dates.get(c)
        on_default = int(c in default_set)

        if SUPPRESS_EMPTY_ROWS and not styles_now:
            if CAPTURE_EMPTY_GAPS and gap_open is None:
                gap_open = {"start_date": dt, "start_commit": c, "prev_state": "+".join(sorted(prev)) if prev else ""}
            continue

        if CAPTURE_EMPTY_GAPS and gap_open is not None and styles_now:
            gap_open["end_date"] = dt
            gap_open["end_commit"] = c
            gap_open["next_state"] = "+".join(sorted(styles_now))
            try:
                d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
                d2 = datetime.fromisoformat((dt or "").replace("Z", "+00:00")).replace(tzinfo=None)
                gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
            except Exception:
                gap_open["duration_days"] = None
            gap_open["censored"] = False
            empty_gaps.append(gap_open)
            gap_open = None

        if styles_now != prev:
            if EMIT_NONE_STATE and not styles_now:
                timeline.append({"date": dt, "commit": c, "on_default": on_default, "styles": [], **meta_now})
            elif styles_now:
                timeline.append({"date": dt, "commit": c, "on_default": on_default, "styles": sorted(styles_now), **meta_now})

            added = styles_now - prev
            removed = prev - styles_now
            for s in sorted(added):
                if s in events:
                    events[s].append({"event": "added", "date": dt, "commit": c, "on_default": on_default})
            for s in sorted(removed):
                if s in events:
                    events[s].append({"event": "removed", "date": dt, "commit": c, "on_default": on_default})
            prev = styles_now

    # Cutoff snapshots (both lenses)
    snapshot_default   = union_snapshot_styles_at_cutoff(repo, default_heads) if default_heads else []
    snapshot_repo_wide = union_snapshot_styles_at_cutoff(repo, all_heads) if all_heads else snapshot_default

    if CAPTURE_EMPTY_GAPS and gap_open is not None:
        # close last gap at cutoff
        gap_open["end_date"] = cutoff_iso
        gap_open["end_commit"] = (default_heads or all_heads or [None])[0]
        gap_open["next_state"] = ""
        try:
            d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
            d2 = datetime.fromisoformat(cutoff_iso.replace("Z", "+00:00")).replace(tzinfo=None)
            gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
        except Exception:
            gap_open["duration_days"] = None
        gap_open["censored"] = True
        empty_gaps.append(gap_open)
        gap_open = None

    off_default_extra = sorted(set(snapshot_repo_wide) - set(snapshot_default))
    qa_issue = None
    if not snapshot_default and not snapshot_repo_wide:
        qa_issue = "No environment detected at cutoff"

    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_default_head": default_heads,
        "cutoff_all_heads": all_heads,
        "first_commit_date": first_commit_date,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": timeline,
        "events": events,
        "snapshot_as_of_cutoff_default": snapshot_default,
        "snapshot_as_of_cutoff_repo_wide": snapshot_repo_wide,
        "off_default_extra_styles": off_default_extra,
        "has_off_default_extra": int(bool(off_default_extra)),
        "empty_gaps": empty_gaps,
        **({"qa_issue": qa_issue} if qa_issue else {}),
    }

# ===========================
# Runner + index/summary
# ===========================
def output_path_for(repo: Path) -> Path:
    return OUTPUT_MINE / f"{repo.name}.emulator_timeline.json"

def list_repos_once(root: Path) -> List[Path]:
    rs = list_repos(root)
    rs.sort(key=lambda p: p.name.lower())
    return rs

def iso_min(dts: List[str]) -> Optional[str]:
    ds = [d for d in dts if d]
    return min(ds) if ds else None

def derive_summary_fields(rec: Dict) -> Dict:
    name = rec.get("repo_name")
    snap_def = rec.get("snapshot_as_of_cutoff_default", [])
    snap_all = rec.get("snapshot_as_of_cutoff_repo_wide", [])
    events = rec.get("events", {})
    first_events: List[Optional[str]] = []
    for k in ("Emu_Community", "Emu_Custom", "GMD", "ThirdParty"):
        for e in events.get(k, []):
            if e.get("event") == "added":
                first_events.append(e.get("date"))
    first_env_date = iso_min([d for d in first_events if d])
    transitions = 0
    for k in events:
        transitions += sum(1 for e in events[k] if e.get("event") in ("added", "removed"))
    gaps = rec.get("empty_gaps", [])
    gap_days = 0.0
    for g in gaps:
        try:
            gap_days += float(g.get("duration_days") or 0.0)
        except Exception:
            pass
    return {
        "repo_name": name,
        "has_env_at_cutoff_default": 1 if snap_def else 0,
        "has_env_at_cutoff_repo_wide": 1 if snap_all else 0,
        "cutoff_states_default": "+".join(snap_def),
        "cutoff_states_repo_wide": "+".join(snap_all),
        "off_default_extra": "+".join(rec.get("off_default_extra_styles", [])),
        "has_off_default_extra": int(bool(rec.get("off_default_extra_styles"))),
        "first_env_date": first_env_date or "",
        "transitions": transitions,
        "total_gap_days": round(gap_days, 2),
        "qa_issue": rec.get("qa_issue", ""),
    }

def run_miner(cutoff_iso: str = CUTOFF_ISO, max_repos: int = MAX_REPOS) -> None:
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)
    # Old:
    #repos = list_repos_once(INPUT_CLONES)
    # New:
    repos = list_repos_hybrid(INPUT_CLONES, PROBLEM_REPO_NAMES)
    print(f"[info] Using hybrid discovery: {len(repos)} repos")
    if max_repos and max_repos > 0:
        repos = repos[:max_repos]

    if not repos:
        print(f"[info] No git repos found under: {INPUT_CLONES}")
        return

    print(f"[info] Work root:         {WORK_ROOT}")
    print(f"[info] Input repos:       {INPUT_CLONES}")
    print(f"[info] Output JSON dir:   {OUTPUT_MINE}")
    print(f"[info] Found repos:       {len(repos)}")
    print(f"[info] Cutoff (UTC):      {cutoff_iso}")
    print(f"[info] Timeline scope:    {TIMELINE_SCOPE}")
    print(f"[info] Options: suppress_empty_rows={SUPPRESS_EMPTY_ROWS}, capture_empty_gaps={CAPTURE_EMPTY_GAPS}, emit_none={EMIT_NONE_STATE}, blob_max_size={BLOB_MAX_SIZE}")

    written = skipped = errors = 0
    summaries: List[Dict] = []

    def process(repo: Path) -> Tuple[str, Path, Optional[Dict], Optional[str]]:
        try:
            outp = output_path_for(repo)
            if RESUME_IF_EXISTS and outp.exists():
                return ("skipped", repo, None, None)
            data = build_timeline(repo, cutoff_iso)
            with outp.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            return ("written", repo, data, None)
        except Exception as e:
            return ("error", repo, None, str(e))

    with PoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(process, r): r for r in repos}
        total = len(futs)
        for i, fut in enumerate(as_completed(futs), 1):
            repo = futs[fut]
            rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
            status, _repo, data, err = fut.result()
            if status == "written":
                print(f"[{i}/{total}] Wrote: {rel}")
                written += 1
                summaries.append(derive_summary_fields(data))  # type: ignore[arg-type]
            elif status == "skipped":
                print(f"[{i}/{total}] Skipped (exists): {rel}")
                skipped += 1
                try:
                    with output_path_for(repo).open("r", encoding="utf-8") as f:
                        data2 = json.load(f)
                    summaries.append(derive_summary_fields(data2))
                except Exception:
                    pass
            else:
                print(f"[{i}/{total}] [error] {rel}: {err}", file=sys.stderr)
                errors += 1

    # Write summary CSV + JSON index
    idx_json = OUTPUT_MINE / "index_summary.json"
    with idx_json.open("w", encoding="utf-8") as f:
        json.dump({"cutoff": cutoff_iso, "repos": summaries}, f, ensure_ascii=False, indent=2)

    idx_csv = OUTPUT_MINE / "index_summary.csv"
    with idx_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "repo_name",
                "has_env_at_cutoff_default",
                "has_env_at_cutoff_repo_wide",
                "cutoff_states_default",
                "cutoff_states_repo_wide",
                "off_default_extra",
                "has_off_default_extra",
                "first_env_date",
                "transitions",
                "total_gap_days",
                "qa_issue",
            ],
        )
        w.writeheader()
        for row in summaries:
            w.writerow(row)

    qa_csv = OUTPUT_MINE / "qa_issues.csv"
    with qa_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["repo_name", "qa_issue"])
        w.writeheader()
        for row in summaries:
            if row.get("qa_issue"):
                w.writerow({"repo_name": row["repo_name"], "qa_issue": row["qa_issue"]})

    print(f"[done] JSON -> {OUTPUT_MINE}  (written={written}, skipped={skipped}, errors={errors})")
    print(f"[done] Summary: {idx_csv.name}, {idx_json.name}, QA: {qa_csv.name}")

if __name__ == "__main__":
    _failfast_checks()
    print("[run] Starting miner…")
    run_miner(cutoff_iso=CUTOFF_ISO, max_repos=MAX_REPOS)
    print("[run] Finished.")


[run] Starting miner…
[info] Using hybrid discovery: 3 repos
[info] Work root:         C:\Temp_Instru
[info] Input repos:       C:\Temp_Instru\clonesV9.0
[info] Output JSON dir:   C:\Temp_Instru\mineV9.3
[info] Found repos:       3
[info] Cutoff (UTC):      2025-08-10 23:59:59 +0000
[info] Timeline scope:    repo_wide
[info] Options: suppress_empty_rows=True, capture_empty_gaps=True, emit_none=False, blob_max_size=2000000


[1/3] [error] rainbow-me__rainbow: [WinError 206] The filename or extension is too long
[2/3] [error] MetaMask__metamask-mobile: [WinError 206] The filename or extension is too long


[done] JSON -> C:\Temp_Instru\mineV9.3  (written=0, skipped=0, errors=3)
[done] Summary: index_summary.csv, index_summary.json, QA: qa_issues.csv
[run] Finished.


[3/3] [error] pytorch__executorch: [WinError 206] The filename or extension is too long
